In [1]:
# %%
# ============================================================
# BranchyNet + Adaptive Computation — ViT (patch4/32) on CIFAR-10
# FIXED VERSION — aligned with branchynet_resnet50_cifar10_fixed.py
#
# Three fixes applied relative to the original BranchyViT script:
#
#   1. TRUE per-sample early exit. adaptive_forward() no longer
#      requires the WHOLE batch to agree before exiting — confident
#      samples are sliced out and never touch deeper transformer
#      blocks, exactly like the fixed BranchyResNet50.
#
#   2. Consistent measurement methodology. disk_size_mb(),
#      compute_flops() (custom hook-based), and measure_inference()
#      are byte-for-byte the same approach as the fixed ResNet
#      script, so ResNet vs ViT numbers are now genuinely comparable.
#      NOTE: compute_flops() only hooks nn.Conv2d / nn.Linear, so it
#      misses the raw Q@K^T / attn@V matmuls inside attention blocks
#      (same blind spot thop has). This still undercounts true ViT
#      FLOPs slightly, but it's now the SAME undercount methodology
#      used for ResNet, so the comparison is apples-to-apples even
#      if neither number is the literal ground truth.
#
#   3. No fine-tuning from an existing baseline checkpoint.
#      BranchyViT now loads ImageNet-pretrained transformer blocks
#      directly (transferring weights + interpolating pos_embed,
#      same strategy as the ViT baseline's build_model()) and trains
#      all three exits jointly from epoch 1 — mirroring how the fixed
#      BranchyResNet50 uses pretrained=True (ImageNet) rather than
#      loading __4__baseline_vit_pretrained_cifar10.pth.
# ============================================================

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import timm
import time, os, json, tempfile, random
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# %%
# ── REPRODUCIBILITY ───────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

# %%
# ── CONFIG ────────────────────────────────────────────────────
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
EPOCHS      = 30
NUM_CLASSES = 10
IMG_SIZE    = 32
PATCH_SIZE  = 4
TIMM_BASE   = 'vit_small_patch16_224'

SAVE_PATH = "__5__branchynet_vit_cifar10_fixed.pth"

EXIT_THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
BRANCH_WEIGHTS  = [0.2, 0.3, 0.5]   # [exit1, exit2, final]

CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"Using device: {DEVICE}")

# %%
# ── UTILITY FUNCTIONS (identical methodology to the fixed ResNet script) ──
def disk_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        size = os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)
    return round(size, 4)

def measure_inference(model, device, img_size=IMG_SIZE, batch_size=128, runs=50):
    m = model.eval().to(device)
    use_cuda = device.type == "cuda"

    def _timed_run(inp, n):
        with torch.no_grad():
            for _ in range(3):
                m(inp)
            if use_cuda:
                torch.cuda.synchronize()
                s = torch.cuda.Event(enable_timing=True)
                e = torch.cuda.Event(enable_timing=True)
                s.record()
                for _ in range(n):
                    m(inp)
                e.record()
                torch.cuda.synchronize()
                return s.elapsed_time(e) / 1000.0
            else:
                t0 = time.perf_counter()
                for _ in range(n):
                    m(inp)
                return time.perf_counter() - t0

    single    = torch.randn(1, 3, img_size, img_size, device=device)
    single_s  = _timed_run(single, runs)
    single_ms = single_s / runs * 1000

    batch    = torch.randn(batch_size, 3, img_size, img_size, device=device)
    batch_s  = _timed_run(batch, runs)
    batch_ms = batch_s / runs * 1000

    return {
        "single_img_ms"      : round(single_ms, 4),
        "batch128_ms"        : round(batch_ms, 4),
        "per_img_ms"         : round(batch_ms / batch_size, 6),
        "throughput_imgs_sec": round(batch_size * runs / batch_s, 2),
        "timing_method"      : "CUDA events" if use_cuda else "perf_counter",
    }

def compute_flops(model, device, input_size=(1, 3, IMG_SIZE, IMG_SIZE)):
    m = model.eval().to(device)
    total_flops = [0]
    hooks = []

    def conv_hook(module, inp, out):
        N, C_out, H_out, W_out = out.shape
        C_in = module.in_channels
        kH, kW = module.kernel_size if isinstance(module.kernel_size, tuple) \
                  else (module.kernel_size, module.kernel_size)
        total_flops[0] += 2 * N * C_out * H_out * W_out * (C_in // module.groups) * kH * kW

    def linear_hook(module, inp, out):
        N = inp[0].shape[0]
        total_flops[0] += 2 * N * module.in_features * module.out_features

    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            hooks.append(mod.register_forward_hook(conv_hook))
        elif isinstance(mod, nn.Linear):
            hooks.append(mod.register_forward_hook(linear_hook))

    with torch.no_grad():
        m(torch.randn(*input_size, device=device))
    for h in hooks:
        h.remove()
    return round(total_flops[0] / 1e9, 6)

# %%
# ── AUXILIARY BRANCH (early-exit classifier head) ─────────────
class EarlyExitBranch(nn.Module):
    """
    Tiny classifier attached to an intermediate ViT hidden state.
    input_dim  : hidden dimension of the tapped transformer layer
    num_classes: number of output classes
    """
    def __init__(self, input_dim: int, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.branch = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        # x: (B, N, D) — take CLS token (index 0) for classification
        cls = x[:, 0]
        return self.branch(cls)

# %%
# ── BRANCHYNET ViT ────────────────────────────────────────────
class BranchyViT(nn.Module):
    """
    ViT-Small (patch4, img32) with two auxiliary early-exit branches
    inserted after block[3] (shallow) and block[7] (mid-depth).
    The final head of the original ViT is the third (deepest) exit.

      Exit 1 → after block  3   (~33% of depth)
      Exit 2 → after block  7   (~67% of depth)
      Exit 3 → after block 11   (full depth, original head)
    """
    def __init__(self, num_classes: int = NUM_CLASSES, img_size: int = IMG_SIZE,
                 patch_size: int = PATCH_SIZE):
        super().__init__()

        # Architecture skeleton only — no ImageNet weights yet.
        # (Avoids a redundant pretrained download when this class is
        # instantiated just to immediately load_state_dict() from an
        # already-trained checkpoint, e.g. for fresh-instance timing.)
        backbone = timm.create_model(
            TIMM_BASE, pretrained=False, num_classes=num_classes,
            img_size=img_size, patch_size=patch_size,
        )

        self.patch_embed = backbone.patch_embed
        self.cls_token   = backbone.cls_token
        self.pos_embed   = backbone.pos_embed
        self.pos_drop    = backbone.pos_drop
        self.blocks      = backbone.blocks      # nn.Sequential of 12 blocks
        self.norm        = backbone.norm
        self.head        = backbone.head

        hidden_dim = backbone.embed_dim          # 384 for vit_small
        self.num_classes = num_classes
        self.img_size    = img_size
        self.patch_size  = patch_size

        self.branch1 = EarlyExitBranch(hidden_dim, num_classes)   # after block 3
        self.branch2 = EarlyExitBranch(hidden_dim, num_classes)   # after block 7

        self.exit1_block_idx = 3
        self.exit2_block_idx = 7

    # ── FIXED: load ImageNet-pretrained backbone directly ──────
    # Transfers all transformer block weights from the ImageNet
    # checkpoint and bilinearly interpolates pos_embed from the
    # 14x14 ImageNet grid to this model's grid (8x8 for patch4/32).
    # patch_embed is left randomly initialised (different kernel
    # size) — same strategy as the ViT baseline's build_model().
    #
    # This REPLACES the old load_backbone(BASELINE_WEIGHTS, ...)
    # call, which fine-tuned from an already-trained CIFAR baseline
    # checkpoint instead of starting from the same ImageNet init the
    # baseline itself started from.
    def load_imagenet_backbone(self, timm_base: str = TIMM_BASE):
        model_pretrained = timm.create_model(
            timm_base, pretrained=True, num_classes=self.num_classes
        )

        own_state    = self.state_dict()
        keys_to_skip = {'patch_embed.proj.weight', 'patch_embed.proj.bias', 'pos_embed'}

        transferred, skipped = 0, 0
        for k, v in model_pretrained.state_dict().items():
            if k in keys_to_skip:
                skipped += 1
                continue
            if k in own_state and own_state[k].shape == v.shape:
                own_state[k] = v
                transferred += 1
        self.load_state_dict(own_state)

        with torch.no_grad():
            old_pos   = model_pretrained.pos_embed         # (1, 197, dim)
            cls_pos   = old_pos[:, :1, :]
            patch_pos = old_pos[:, 1:, :]
            dim       = patch_pos.shape[-1]
            old_grid  = int(patch_pos.shape[1] ** 0.5)      # 14
            new_grid  = self.img_size // self.patch_size    # 8

            patch_pos = patch_pos.reshape(1, old_grid, old_grid, dim).permute(0, 3, 1, 2)
            patch_pos = torch.nn.functional.interpolate(
                patch_pos, size=(new_grid, new_grid), mode='bilinear', align_corners=False
            )
            patch_pos = patch_pos.permute(0, 2, 3, 1).reshape(1, new_grid * new_grid, dim)

            new_pos = torch.cat([cls_pos, patch_pos], dim=1)
            self.pos_embed.copy_(new_pos)

        print(f"  ImageNet backbone transferred  : {transferred} tensors")
        print(f"  Re-initialised (shape mismatch): {skipped} tensors "
              f"(patch_embed + pos_embed; branch1/branch2 stay randomly init)")

    # ── Full forward — all three exits (used during training) ──
    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)                                # (B, 64, D)
        cls = self.cls_token.expand(B, -1, -1)                 # (B,  1, D)
        x   = torch.cat([cls, x], dim=1)                       # (B, 65, D)
        x   = self.pos_drop(x + self.pos_embed)

        for i, blk in enumerate(self.blocks):
            x = blk(x)
            if i == self.exit1_block_idx:
                logits1 = self.branch1(x)
            elif i == self.exit2_block_idx:
                logits2 = self.branch2(x)

        x       = self.norm(x)
        logits3 = self.head(x[:, 0])

        return logits1, logits2, logits3

    # ── FIXED: TRUE per-sample early exit ──────────────────────
    # Confident samples are sliced out of the running batch and
    # never execute deeper blocks — only unresolved samples continue.
    # This is the same fix applied to BranchyResNet50.adaptive_forward,
    # adapted to slice the (B, N, D) token tensor along the batch dim
    # instead of a (B, C, H, W) feature map.
    @torch.no_grad()
    def adaptive_forward(self, imgs, threshold: float = 0.8):
        """
        Returns
        -------
        final_logits : Tensor (B, num_classes)
        exit_idx     : Tensor (B,) — 0 = branch1, 1 = branch2, 2 = final
        """
        B      = imgs.shape[0]
        device = imgs.device

        final_logits     = torch.zeros(B, self.num_classes, device=device)
        exit_idx         = torch.full((B,), 2, dtype=torch.long, device=device)
        remaining_global  = torch.arange(B, device=device)

        h   = self.patch_embed(imgs)
        cls = self.cls_token.expand(B, -1, -1)
        h   = torch.cat([cls, h], dim=1)
        h   = self.pos_drop(h + self.pos_embed)

        # ── Blocks 0–exit1 ───────────────────────────────────
        for i in range(self.exit1_block_idx + 1):
            h = self.blocks[i](h)

        logits1 = self.branch1(h)
        conf1   = torch.softmax(logits1, dim=1).max(dim=1).values
        done1   = conf1 >= threshold
        if done1.any():
            g_idx = remaining_global[done1]
            final_logits[g_idx] = logits1[done1]
            exit_idx[g_idx]     = 0

        keep = ~done1
        if not keep.any():
            return final_logits, exit_idx

        h                = h[keep]
        remaining_global = remaining_global[keep]

        # ── Blocks exit1+1–exit2 ─────────────────────────────
        for i in range(self.exit1_block_idx + 1, self.exit2_block_idx + 1):
            h = self.blocks[i](h)

        logits2 = self.branch2(h)
        conf2   = torch.softmax(logits2, dim=1).max(dim=1).values
        done2   = conf2 >= threshold
        if done2.any():
            g_idx = remaining_global[done2]
            final_logits[g_idx] = logits2[done2]
            exit_idx[g_idx]     = 1

        keep = ~done2
        if not keep.any():
            return final_logits, exit_idx

        h                = h[keep]
        remaining_global = remaining_global[keep]

        # ── Remaining blocks → final head ────────────────────
        for i in range(self.exit2_block_idx + 1, len(self.blocks)):
            h = self.blocks[i](h)

        h       = self.norm(h)
        logits3 = self.head(h[:, 0])

        final_logits[remaining_global] = logits3
        # exit_idx already = 2 for these samples (default fill value)

        return final_logits, exit_idx


# %%
# ── BUILD MODEL — ImageNet-pretrained backbone, jointly trained ─
# No baseline checkpoint is loaded here. BranchyViT starts from the
# same ImageNet init as the ViT baseline and trains all three exits
# from epoch 1, mirroring BranchyResNet50(pretrained=True).
model = BranchyViT(num_classes=NUM_CLASSES, img_size=IMG_SIZE, patch_size=PATCH_SIZE)
model.load_imagenet_backbone()
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters (BranchyViT): {total_params:,}")

# %%
# ── DATA ─────────────────────────────────────────────────────
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    transforms.RandomErasing(p=0.25),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_set = torchvision.datasets.CIFAR10(root='../data', train=True,
                                          download=True, transform=transform_train)
test_set  = torchvision.datasets.CIFAR10(root='../data', train=False,
                                          download=True, transform=transform_test)

train_loader = torch.utils.data.DataLoader(
    train_set, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, worker_init_fn=seed_worker, generator=g, pin_memory=True,
)
test_loader = torch.utils.data.DataLoader(
    test_set, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True,
)

print(f"Train: {len(train_set)} | Test: {len(test_set)}")

# %%
# ── TRAINING ──────────────────────────────────────────────────
WARMUP_EPOCHS = 3
LR            = 3e-4

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

def branchynet_loss(logits_list, labels, weights=BRANCH_WEIGHTS):
    assert len(logits_list) == len(weights)
    return sum(w * criterion(l, labels) for w, l in zip(weights, logits_list))

# Three LR groups:
#   patch_embed — re-initialised (different kernel size)   → highest LR
#   branch1/2   — newly added, randomly initialised heads   → higher LR
#   everything else — pretrained ImageNet transformer blocks → base LR
patch_embed_params = list(model.patch_embed.parameters())
patch_embed_ids    = {id(p) for p in patch_embed_params}

branch_params = list(model.branch1.parameters()) + list(model.branch2.parameters())
branch_ids    = {id(p) for p in branch_params}

other_params = [p for p in model.parameters()
                if id(p) not in patch_embed_ids and id(p) not in branch_ids]

optimizer = torch.optim.AdamW([
    {'params': patch_embed_params, 'lr': LR * 10},   # re-initialised
    {'params': branch_params,      'lr': LR * 5},    # randomly init heads
    {'params': other_params,       'lr': LR},        # pretrained backbone
], weight_decay=0.05)

def get_lr(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / (EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr)


def train_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits1, logits2, logits3 = model(inputs)
        loss = branchynet_loss([logits1, logits2, logits3], labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        correct    += logits3.argmax(1).eq(labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total


def evaluate_standard(model, loader):
    """Accuracy using only the final exit head."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            _, _, logits3 = model(inputs)
            correct += logits3.argmax(1).eq(labels).sum().item()
            total   += labels.size(0)
    return correct / total


best_val_acc = 0.0
train_accs, val_accs, train_losses = [], [], []

print("\n" + "="*60)
print("TRAINING — BranchyViT (3 exits)")
print("="*60)

for epoch in range(EPOCHS):
    loss, train_acc = train_epoch(model, train_loader, optimizer)
    val_acc         = evaluate_standard(model, test_loader)
    scheduler.step()

    current_lr = optimizer.param_groups[-1]['lr']
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    train_losses.append(loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        marker = " ← best saved"
    else:
        marker = ""

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | LR: {current_lr:.6f} | Loss: {loss:.4f} | "
          f"Train: {train_acc:.4f} | Val: {val_acc:.4f}{marker}")

print(f"\nBest validation accuracy (final exit): {best_val_acc:.4f} ({best_val_acc*100:.2f}%)")

# %%
# ── FULL EVALUATION (standard — final exit only) ──────────────
print("\n" + "="*60)
print("FULL EVALUATION — final exit (no early stopping)")
print("="*60)

model = BranchyViT(num_classes=NUM_CLASSES, img_size=IMG_SIZE, patch_size=PATCH_SIZE).to(DEVICE)
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        _, _, logits3 = model(inputs)
        all_preds.extend(logits3.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc_final  = accuracy_score(all_labels, all_preds)
prec_final = precision_score(all_labels, all_preds, average='macro', zero_division=0)
rec_final  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
f1_final   = f1_score(all_labels, all_preds, average='macro', zero_division=0)

print(f"  Accuracy          : {acc_final:.4f}")
print(f"  Precision (macro) : {prec_final:.4f}")
print(f"  Recall    (macro) : {rec_final:.4f}")
print(f"  F1-score  (macro) : {f1_final:.4f}")

# %%
# ── ADAPTIVE COMPUTATION EVALUATION ──────────────────────────
print("\n" + "="*60)
print("ADAPTIVE COMPUTATION — Early-Exit Threshold Sweep")
print("="*60)
print(f"  Thresholds tested: {EXIT_THRESHOLDS}")
print("  NOTE: avg_time_ms below includes dataloader/decode overhead")
print("  (same caveat as the fixed ResNet script's threshold sweep) —")
print("  treat inference_ms_adaptive_tau08 (isolated, below) as the")
print("  trustworthy single-image latency number.")

adaptive_results = []

for tau in EXIT_THRESHOLDS:
    preds_list, labels_list, exit_idx_list = [], [], []

    t_start = time.time()
    model.eval()
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(DEVICE)
            logits, exit_idx = model.adaptive_forward(inputs, threshold=tau)
            preds_list.extend(logits.argmax(1).cpu().numpy())
            labels_list.extend(labels.numpy())
            exit_idx_list.extend(exit_idx.cpu().numpy())
    t_end = time.time()

    preds_arr    = np.array(preds_list)
    labels_arr   = np.array(labels_list)
    exit_idx_arr = np.array(exit_idx_list)
    n            = len(labels_arr)

    acc   = accuracy_score(labels_arr, preds_arr)
    prec  = precision_score(labels_arr, preds_arr, average='macro', zero_division=0)
    rec   = recall_score(labels_arr, preds_arr, average='macro', zero_division=0)
    f1    = f1_score(labels_arr, preds_arr, average='macro', zero_division=0)

    frac_exit1  = (exit_idx_arr == 0).mean()
    frac_exit2  = (exit_idx_arr == 1).mean()
    frac_exit3  = (exit_idx_arr == 2).mean()
    avg_time_ms = (t_end - t_start) / n * 1000

    adaptive_results.append({
        "threshold"  : tau,
        "accuracy"   : round(acc,  6),
        "precision"  : round(prec, 6),
        "recall"     : round(rec,  6),
        "f1"         : round(f1,   6),
        "frac_exit1" : round(frac_exit1, 4),
        "frac_exit2" : round(frac_exit2, 4),
        "frac_exit3" : round(frac_exit3, 4),
        "avg_time_ms": round(avg_time_ms, 4),
    })

    print(f"  τ={tau:.2f} | Acc={acc:.4f} | F1={f1:.4f} | "
          f"Exit1={frac_exit1:.1%} Exit2={frac_exit2:.1%} Exit3={frac_exit3:.1%} | "
          f"Time={avg_time_ms:.4f}ms/sample")

# %%
# ── MODEL COMPLEXITY METRICS ──────────────────────────────────
print("\n" + "="*60)
print("MODEL COMPLEXITY METRICS")
print("="*60)

flops_g = compute_flops(model, DEVICE)
infer   = measure_inference(model, DEVICE)
size_mb = disk_size_mb(model)

# Adaptive inference timing at τ=0.8 (CUDA events, single image) —
# same inline pattern as the fixed ResNet script.
use_cuda     = DEVICE.type == "cuda"
dummy_single = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
with torch.no_grad():
    for _ in range(3):
        model.adaptive_forward(dummy_single, threshold=0.8)
    if use_cuda:
        torch.cuda.synchronize()
        start_evt = torch.cuda.Event(enable_timing=True)
        end_evt   = torch.cuda.Event(enable_timing=True)
        start_evt.record()
        for _ in range(50):
            model.adaptive_forward(dummy_single, threshold=0.8)
        end_evt.record()
        torch.cuda.synchronize()
        inf_ms_adapt = start_evt.elapsed_time(end_evt) / 50
    else:
        t0 = time.perf_counter()
        for _ in range(50):
            model.adaptive_forward(dummy_single, threshold=0.8)
        inf_ms_adapt = (time.perf_counter() - t0) / 50 * 1000

print(f"  Parameters               : {total_params:,}")
print(f"  Model size               : {size_mb:.4f} MB")
print(f"  FLOPs (full, all exits)  : {flops_g:.4f} G")
print(f"  Inference (single, full) : {infer['single_img_ms']:.2f} ms")
print(f"  Inference (batch128)     : {infer['batch128_ms']:.2f} ms "
      f"({infer['throughput_imgs_sec']:.0f} img/s)")
print(f"  Inference (τ=0.8 adapt)  : {inf_ms_adapt:.3f} ms")

# %%
# ── SAVE METRICS JSON ─────────────────────────────────────────
# Top-level schema identical to branchynet_resnet50_cifar10_fixed.py's
# __3__branchynet_metrics.json (method/variant/dataset/accuracy/.../
# inference_ms/num_exits/exit_positions/branch_weights/
# exit_thresholds_tested/inference_ms_adaptive_tau08/
# adaptive_threshold_results/best_adaptive_result/saved_model_path),
# with ViT-only metadata appended at the end.
best_adaptive = max(adaptive_results, key=lambda r: r["accuracy"])

branchynet_vit_metrics = {
    "method"                      : "early_exit",
    "variant"                     : "BranchyViT_Small",
    "dataset"                     : "CIFAR-10",
    "accuracy"                    : round(acc_final,  6),
    "precision"                   : round(prec_final, 6),
    "recall"                      : round(rec_final,  6),
    "f1"                          : round(f1_final,   6),
    "params"                      : total_params,
    "size_mb"                     : size_mb,
    "flops_G"                     : flops_g,
    "inference_ms"                : infer,
    "num_exits"                   : 3,
    "exit_positions"              : [
        f"after block {model.exit1_block_idx} of 11 (~33% depth)",
        f"after block {model.exit2_block_idx} of 11 (~67% depth)",
        "final head (full depth)",
    ],
    "branch_weights"              : BRANCH_WEIGHTS,
    "exit_thresholds_tested"      : EXIT_THRESHOLDS,
    "inference_ms_adaptive_tau08" : round(inf_ms_adapt, 4),
    "adaptive_threshold_results"  : adaptive_results,
    "best_adaptive_result"        : best_adaptive,
    # ── ViT-specific metadata (no ResNet equivalent) ───────────
    "input_resolution"            : IMG_SIZE,
    "patch_size"                  : PATCH_SIZE,
    "num_patches"                 : (IMG_SIZE // PATCH_SIZE) ** 2,
    "original_arch"               : TIMM_BASE,
    "init_strategy"                : ("ImageNet-pretrained backbone, jointly trained "
                                      "(no fine-tune from baseline checkpoint)"),
    "saved_model_path"            : os.path.abspath(SAVE_PATH),
}

output_json = "__5__branchynet_vit_metrics.json"
with open(output_json, "w") as f:
    json.dump(branchynet_vit_metrics, f, indent=2)
print(f"\n✓ Metrics saved to {output_json}")

# %%
# ── COMPARISON SUMMARY ────────────────────────────────────────
print("\n" + "="*60)
print("COMPARISON SUMMARY: ViT Baseline vs BranchyViT")
print("="*60)

baseline_path = "__4__baseline_metrics.json"
if os.path.exists(baseline_path):
    with open(baseline_path) as f:
        base = json.load(f)

    scalar_keys = ["accuracy", "precision", "recall", "f1", "params", "size_mb", "flops_G"]
    print(f"\n  {'Metric':<22} {'Baseline':>12} {'BranchyViT':>12} {'Δ':>10}")
    print("  " + "-"*58)
    for k in scalar_keys:
        bv = base.get(k, float('nan'))
        mv = branchynet_vit_metrics.get(k, float('nan'))
        if not isinstance(bv, (int, float)) or not isinstance(mv, (int, float)):
            continue
        delta = mv - bv
        sign  = "+" if delta > 0 else ""
        if k == "params":
            print(f"  {k:<22} {bv:>12,} {mv:>12,} {sign}{delta:>9,.0f}")
        else:
            print(f"  {k:<22} {bv:>12.4f} {mv:>12.4f} {sign}{delta:>9.4f}")

    # NOTE: the original ViT baseline script (__4__baseline_metrics.json)
    # saves "inference_ms" as a single float (single-image only), not the
    # {single_img_ms, batch128_ms, ...} dict the fixed scripts use. This
    # block handles both formats so it won't crash either way — but it's
    # worth fixing the baseline script too if you want full consistency.
    base_inf = base.get("inference_ms")
    if isinstance(base_inf, dict):
        base_single = base_inf.get("gpu", base_inf).get("single_img_ms")
    else:
        base_single = base_inf  # old format: plain float

    full_ms = infer['single_img_ms']
    if base_single is not None:
        speedup = (1 - full_ms / base_single) * 100
        print(f"\n  Inference single, full — Baseline  : {base_single:.3f} ms")
        print(f"  Inference single, full — BranchyViT: {full_ms:.3f} ms  ({speedup:+.1f}%)")
    print(f"  Inference (τ=0.8 adapt)            : {inf_ms_adapt:.3f} ms  "
          f"(vs full single: {(1 - inf_ms_adapt/full_ms)*100:+.1f}%)")
    print(f"\n  Best adaptive result (τ={best_adaptive['threshold']}):")
    print(f"    Accuracy   : {best_adaptive['accuracy']:.4f}")
    print(f"    Exit1      : {best_adaptive['frac_exit1']:.1%}")
    print(f"    Exit2      : {best_adaptive['frac_exit2']:.1%}")
    print(f"    Exit3      : {best_adaptive['frac_exit3']:.1%}")
    print(f"    Time/sample: {best_adaptive['avg_time_ms']:.4f} ms")
else:
    print(f"  (baseline JSON not found at {baseline_path} — skipping diff)")

# %%
# ── SAVE FULL CHECKPOINT ──────────────────────────────────────
config_dict = {
    "model_name"    : "BranchyViT_Small_CIFAR10_fixed",
    "timm_base"     : TIMM_BASE,
    "img_size"      : IMG_SIZE,
    "patch_size"    : PATCH_SIZE,
    "num_patches"   : (IMG_SIZE // PATCH_SIZE) ** 2,
    "num_classes"   : NUM_CLASSES,
    "num_exits"     : 3,
    "exit_positions": [
        f"after block {model.exit1_block_idx}",
        f"after block {model.exit2_block_idx}",
        "final head",
    ],
    "branch_weights": BRANCH_WEIGHTS,
    "normalization" : {"mean": CIFAR_MEAN, "std": CIFAR_STD},
    "init_strategy" : "ImageNet-pretrained backbone, jointly trained from epoch 1",
    "training": {
        "batch_size"      : BATCH_SIZE,
        "epochs"          : EPOCHS,
        "learning_rate"   : LR,
        "patch_embed_lr"  : LR * 10,
        "branch_lr"       : LR * 5,
        "optimizer"       : "AdamW",
        "weight_decay"    : 0.05,
        "scheduler"       : "WarmupCosine",
        "warmup_epochs"   : WARMUP_EPOCHS,
        "grad_clip"       : 1.0,
        "label_smoothing" : 0.1,
    }
}

torch.save({
    "model_state_dict": model.state_dict(),
    "config"          : config_dict,
    "classes"         : CIFAR10_CLASSES,
}, "__5__model_checkpoint_fixed.pth")

print("\n" + "="*60)
print("BRANCHYNET ViT + ADAPTIVE COMPUTATION — COMPLETE (FIXED)")
print("="*60)
print(f"  Weights    → {SAVE_PATH}")
print(f"  Checkpoint → __5__model_checkpoint_fixed.pth")
print(f"  Metrics    → {output_json}")

e:\baseline_ViT\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


  ImageNet backbone transferred  : 149 tensors
  Re-initialised (shape mismatch): 3 tensors (patch_embed + pos_embed; branch1/branch2 stay randomly init)
Total parameters (BranchyViT): 21,445,022


e:\baseline_ViT\env\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Train: 50000 | Test: 10000

TRAINING — BranchyViT (3 exits)
Epoch  1/30 | LR: 0.000200 | Loss: 1.7568 | Train: 0.4292 | Val: 0.6426 ← best saved
Epoch  2/30 | LR: 0.000300 | Loss: 1.2976 | Train: 0.6794 | Val: 0.7974 ← best saved
Epoch  3/30 | LR: 0.000300 | Loss: 1.0805 | Train: 0.7951 | Val: 0.8677 ← best saved
Epoch  4/30 | LR: 0.000299 | Loss: 0.9505 | Train: 0.8508 | Val: 0.8853 ← best saved
Epoch  5/30 | LR: 0.000296 | Loss: 0.8762 | Train: 0.8819 | Val: 0.8986 ← best saved
Epoch  6/30 | LR: 0.000291 | Loss: 0.8325 | Train: 0.8967 | Val: 0.9056 ← best saved
Epoch  7/30 | LR: 0.000284 | Loss: 0.7928 | Train: 0.9125 | Val: 0.9171 ← best saved
Epoch  8/30 | LR: 0.000275 | Loss: 0.7673 | Train: 0.9215 | Val: 0.9243 ← best saved
Epoch  9/30 | LR: 0.000265 | Loss: 0.7468 | Train: 0.9297 | Val: 0.9280 ← best saved
Epoch 10/30 | LR: 0.000253 | Loss: 0.7244 | Train: 0.9394 | Val: 0.9339 ← best saved
Epoch 11/30 | LR: 0.000240 | Loss: 0.7035 | Train: 0.9450 | Val: 0.9260
Epoch 12/30 | LR: 